<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/GLM_2_regi%C3%B3n_agencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

equipo	N	m_sin	disp_sin	p0_fraude	p_evento	p_baja	m_coste
equipo_07	260107	1.1	1.1	1	1.1	0.7	1.7	1.05	0.56


In [ ]:
library(tidyverse)

# Fuente de datos del equipo 07
repositorio <- "https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/"

cartera <- read_csv(
    paste0(repositorio, "cartera.csv"),
    show_col_types = FALSE
    )

# Comprobar que los datos son válidos
stopifnot(
    nrow(cartera) > 0,
    all(c("n_siniestros", "n_asistencia", "exposicion")    %in% names(cartera)),
    all(!is.na(cartera$exposicion)),
    all(cartera$exposicion > 0)
    )

glimpse(cartera)
summary(cartera$exposicion)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Rows: 3,062
Columns: 40
$ id_poliza           <dbl> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,…
$ id_agencia          <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ id_region           <dbl> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,…
$ antiguedad          <dbl> 1.082, 6.687, 5.732, 1.397, 1.968, 3.159, 3.821, 7…
$ exposicion          <dbl> 0.1082, 0.6687, 0.5732, 0.1397, 0.1968, 0.3159, 0.…
$ edad_conductor      <dbl> 33, 44, 62, 25, 66, 25, 18, 44, 49, 36, 77, 36, 39…
$ edad_tramo          <chr> "25-35", "35-65", "35-65", "18-25", "65+", "18-25"…
$ antiguedad_carnet   <dbl> 7, 13, 36, 6, 36, 7, 0, 11, 30, 7, 49, 12, 16, 17,…
$ potencia_cv         <dbl> 85, 142, 118, 76, 55, 120, 143, 64, 113, 136, 141,…
$ cilindrada          <dbl> 1417, 2423, 1831, 1278, 824, 1816, 1730, 1006, 162…
$ antiguedad_vehiculo <dbl> 6, 12, 6, 9, 10, 6, 11, 10, 19, 4, 7, 12, 19, 4, 7…
$ valor_vehiculo      <dbl> 35790, 22089, 31528, 25948, 20714, 15821, 23040, 2…
$ densidad      

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0100  0.2194  0.3768  0.4207  0.5934  0.9996 

## Gestiones por región y agencia: violín y vela

**Respuesta:** \`n_gestiones\` es el número de gestiones observado en cada póliza. Queremos comparar su **distribución**, primero entre regiones y después entre agencias de cada región, sin ajustar todavía un GLM.

- **Violín:** densidad suavizada de los conteos (los valores intermedios de la silueta *no* son gestiones fraccionarias observadas). Todos los grupos tienen la misma anchura máxima: la anchura no mide cuántas pólizas contiene un grupo.
- **Vela interior:** caja de Q1 a Q3, línea central de la mediana y bigotes hasta 1,5 veces el rango intercuartílico. Los puntos atípicos no se dibujan para evitar saturación, pero sí participan en los cálculos.
- **Color:** identifica la región, no un resultado más favorable o desfavorable.
- **Limitación:** \`n_gestiones\` es un conteo *bruto*. Con una exposición distinta, dos pólizas pueden acumular un número distinto de gestiones incluso si comparten la misma intensidad por unidad de exposición. Los efectos ajustados se estudiarán posteriormente con un modelo de conteos y \`offset(log(exposicion))\`.

La construcción gráfica reproduce el esquema del Caso 1: \`geom_violin()\` + \`geom_boxplot()\`.

In [ ]:
# 1. Selección de variables y comprobaciones
stopifnot(
  all(c("id_region", "id_agencia", "n_gestiones", "exposicion") %in% names(cartera))
)

datos_gestiones <- cartera %>%
  filter(!is.na(id_region), !is.na(id_agencia), !is.na(n_gestiones)) %>%
  mutate(
    region = factor(id_region),
    agencia = factor(id_agencia)
  )

stopifnot(
  nrow(datos_gestiones) > 0,
  all(is.finite(datos_gestiones$n_gestiones)),
  all(datos_gestiones$n_gestiones >= 0),
  all(datos_gestiones$n_gestiones == floor(datos_gestiones$n_gestiones))
)

# Tamaño de cada grupo: necesario para interpretar las figuras
tamano_agencia <- datos_gestiones %>%
  count(region, agencia, name = "n_polizas_agencia")

datos_gestiones <- datos_gestiones %>%
  left_join(tamano_agencia, by = c("region", "agencia"))

paleta_regiones <- setNames(
  grDevices::hcl.colors(nlevels(datos_gestiones$region), palette = "Dark 3"),
  levels(datos_gestiones$region)
)

cat("Regiones:", nlevels(datos_gestiones$region),
    "| Agencias por región:", nrow(tamano_agencia),
    "| Pólizas válidas:", nrow(datos_gestiones), "\n")
cat("Registros omitidos por valores ausentes:",
    nrow(cartera) - nrow(datos_gestiones), "\n")

# El número de agencias por región y sus tamaños
tamano_agencia %>%
  group_by(region) %>%
  summarise(
    agencias = n(),
    polizas = sum(n_polizas_agencia),
    menor_agencia = min(n_polizas_agencia),
    .groups = "drop"
  ) %>% print(n = Inf)

In [ ]:
# 2. Distribución de gestiones por región
grafico_regiones <- ggplot(
  datos_gestiones,
  aes(x = region, y = n_gestiones, fill = region)
) +
  geom_violin(
    trim = TRUE, scale = "width",
    alpha = 0.35, linewidth = 1.0, colour = "grey25"
  ) +
  geom_boxplot(
    width = 0.18, alpha = 0.95,
    fill = "white", colour = "grey15",
    linewidth = 0.8, outlier.shape = NA
  ) +
  scale_fill_manual(values = paleta_regiones) +
  labs(
    title = "Número de gestiones por región",
    subtitle = "Distribución por póliza: violín con vela (caja) interior",
    x = "Región", y = "Número de gestiones por póliza"
  ) +
  guides(fill = "none") +
  theme_minimal(base_size = 12)

print(grafico_regiones)

In [ ]:
# 3. Distribución de gestiones por agencia, identificadas por región
# Las agencias pequeñas conservan su caja, incluso si no permiten
# estimar de forma fiable un violín (menos de 3 observaciones).
grafico_agencias <- ggplot(
  datos_gestiones,
  aes(x = agencia, y = n_gestiones, fill = region)
) +
  geom_violin(
    data = filter(datos_gestiones, n_polizas_agencia >= 3),
    trim = TRUE, scale = "width",
    alpha = 0.35, linewidth = 0.8, colour = "grey25"
  ) +
  geom_boxplot(
    width = 0.18, alpha = 0.95,
    fill = "white", colour = "grey15",
    linewidth = 0.7, outlier.shape = NA
  ) +
  facet_wrap(~ region, scales = "free_x", ncol = 2) +
  scale_fill_manual(values = paleta_regiones) +
  labs(
    title = "Número de gestiones por agencia",
    subtitle = "Agencias agrupadas y coloreadas según su región",
    x = "Agencia", y = "Número de gestiones por póliza",
    fill = "Región"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 65, hjust = 1),
    legend.position = "bottom"
  )

print(grafico_agencias)

### Lectura antes del GLM

1. **Comparar centros:** ¿qué regiones o agencias tienen medianas diferentes? Cuando muchos recuentos son cero, la mediana podría coincidir incluso si el resto de distribuciones no.
2. **Comparar dispersión y colas:** ¿hay agencias con gran variabilidad en gestiones? El violín permite ver la forma aproximada; el bigote no es un intervalo de confianza.
3. **Separar dos niveles:** una diferencia de agencias dentro de una región no demuestra por sí sola un efecto de agencia; las pólizas pueden tener exposiciones y perfiles distintos.

**Siguiente etapa, todavía no realizada:** estudiar un modelo de Poisson para \`n_gestiones\`, incorporando la exposición con \`offset(log(exposicion))\`, y comparar variabilidad de región frente a agencia.